# Dataset

In [ ]:
import pandas

df = pandas.read_csv("https://raw.githubusercontent.com/stedy/Machine-Learning-with-R-datasets/master/insurance.csv")
#print(df)

cat_cols = ["children", "sex", "smoker", "region"]
cont_cols = ["age", "bmi"]

X = df[cat_cols + cont_cols]
y = df["charges"]

# Workflow

## Default base estimator

In [ ]:
from ngboost.learners import default_tree_learner
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

transformer = ColumnTransformer([
    ("cat", OneHotEncoder(sparse_output = False), cat_cols),
    ("cont", "passthrough", cont_cols)
])

base_estimator = default_tree_learner

## XGBoost base estimator

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBRegressor

transformer = ColumnTransformer([
    ("cat", OrdinalEncoder(), cat_cols),
    ("cont", "passthrough", cont_cols)
])

feature_types = ["c"] * len(cat_cols) + ["q"] * len(cont_cols)

# Relatively short and shallow configuration
base_estimator = XGBRegressor(n_estimators = 3, max_depth = 2, feature_types = feature_types)

## Pipeline

In [ ]:
from ngboost import NGBRegressor
from ngboost.distns import LogNormal
from sklearn2pmml.pipeline import PMMLPipeline

regressor = NGBRegressor(Base = base_estimator, Dist = LogNormal, n_estimators = 111, learning_rate = 0.1, random_state = 42)

pipeline = PMMLPipeline([
    ("transformer", transformer),
    ("regressor", regressor)
])
pipeline.fit(X, y)

# Mark the regressor as fitted
pipeline["regressor"].fitted_ = True

# Export to PMML

In [ ]:
from sklearn2pmml import sklearn2pmml

sklearn2pmml(pipeline, "NGBoostPipeline.pmml")

## Distribution parameters

In [ ]:
from sklearn.pipeline import make_pipeline, FeatureUnion
from sklearn2pmml import sklearn2pmml
from sklearn2pmml.cross_reference import Recaller
from sklearn2pmml.decoration import Alias
from sklearn2pmml.preprocessing import ExpressionTransformer    

def make_branch(in_name, expr, out_name):
    recaller = Recaller(memory = None, names = [in_name])
    transformer = Alias(ExpressionTransformer(expr), name = out_name, prefit = True)
    return make_pipeline(recaller, transformer)

dist_output = FeatureUnion([
    ("loc", make_branch("loc", "X['loc'] + 0.0", "dist(mu)")),
    ("scale", make_branch("scale", "numpy.exp(X['scale'])", "dist(sigma)"))
])

pipeline.predict_transformer = dist_output

sklearn2pmml(pipeline, "NGBoostPipeline-dist.pmml")

## Prediction intervals

In [ ]:
from sklearn2pmml import sklearn2pmml

pipeline.configure(confidence_level = 0.90)
sklearn2pmml(pipeline, "NGBoostPipeline-static.pmml")

pipeline.configure(confidence_level = "ci")
sklearn2pmml(pipeline, "NGBoostPipeline-dynamic.pmml")

In [ ]:
from jpmml_evaluator import make_evaluator

evaluator = make_evaluator("NGBoostPipeline-dynamic.pmml") \
	.verify()

pmml_X = X.copy()
#pmml_X["ci"] = 0.75
pmml_X["ci"] = 0.90

pmml_y = evaluator.evaluateAll(pmml_X)
print(pmml_y)